# LoRA Fine-Tuning for Sequence Classification
This notebook follows the checkpoints in the README to fine-tune BERT using LoRA.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import torch 
from peft import LoraConfig, get_peft_model
import evaluate 

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load Dataset (e.g., dair-ai/emotion)
dataset = load_dataset("dair-ai/emotion")
emotion_labels = dataset["train"].features["label"].names
print("Labels:", emotion_labels)

In [ ]:
# Tokenization & Preprocessing
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocessor(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

# Map preprocessor to dataset
dataset = dataset.map(preprocessor, batched=True)
if "label" in dataset["train"].column_names:
    dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# Model Setup
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)
model.to(device)

# Applying LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
# Training Setup
args = TrainingArguments(
    output_dir="./lora_emotion",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    num_train_epochs=2,    
    eval_strategy="epoch",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

In [ ]:
# Train Model
trainer.train()

In [ ]:
# Save the Adapter
model.save_pretrained("./lora_emotion_adapter")
print("LoRA fine-tuning complete! Adapter saved at ./lora_emotion_adapter")

In [ ]:
# Test the Model on Custom Sentences
samples = [
    "I am so happy to see you!",
    "This is terrifying, I can't handle it.",
    "He surprised everyone with his gift.",
    "I feel so sad and lonely.",
    "I love spending time with my family."
]

inputs = tokenizer(samples, truncation=True, padding=True, return_tensors="pt").to(device)

model.eval()
with torch.no_grad():
    logits = model(**inputs).logits
    preds = torch.argmax(logits, dim=-1)

for text, pred in zip(samples, preds):
    print(f"Text: {text}\nPredicted Emotion: {emotion_labels[pred]}\n")